# Cyril: offline fingerprint MLP submission


In [ ]:
# Offline inference needs no credentials.


In [ ]:
from pathlib import Path
EXPERIMENT = 'E102'  # E101, E102, or E103
# Set explicit roots if the automatic unique-match discovery is ambiguous.
WEIGHTS = None
SOURCE = None
TEST_PATH = None
WHEELS = None  # e.g. Path('/kaggle/input/casmi-wheels')
assert EXPERIMENT in {'E101', 'E102', 'E103'}

In [ ]:
import sys, json, subprocess, hashlib, importlib.util
import numpy as np
import pandas as pd

def unique_match(pattern):
    matches = list(Path('/kaggle/input').glob(pattern))
    if len(matches) != 1:
        raise ValueError(f'Expected one {pattern}; found {matches}. Configure an explicit path.')
    return matches[0]

WEIGHTS = Path(WEIGHTS) if WEIGHTS else unique_match('**/checksums.json').parent
SOURCE = Path(SOURCE) if SOURCE else unique_match('**/casmi_src')
TEST_PATH = Path(TEST_PATH) if TEST_PATH else unique_match('**/test.parquet')
if importlib.util.find_spec('rdkit') is None:
    assert WHEELS is not None, 'Attach compatible RDKit wheels and set WHEELS.'
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', str(WHEELS), 'rdkit'])
import rdkit, torch
sys.path.insert(0, str(SOURCE))

def digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

checksums = json.loads((WEIGHTS / 'checksums.json').read_text())
for name in ['model_mlp.py', 'E101.pt', 'E102.pt', 'selection.json']:
    assert digest(WEIGHTS / name) == checksums[name], f'Checksum mismatch: {name}'
spec = importlib.util.spec_from_file_location('cyril_model_mlp', WEIGHTS / 'model_mlp.py')
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
from src.data.build_reference import build
from src.evaluation.validate import make_submission
selection = json.loads((WEIGHTS / 'selection.json').read_text())
chosen = selection['model'] if EXPERIMENT == 'E103' else EXPERIMENT
checkpoint = WEIGHTS / f'{chosen}.pt'
if EXPERIMENT == 'E103':
    assert digest(checkpoint) == selection['checkpoint_sha256']
model = module.FingerprintMLP(checkpoint, device='cpu')
if EXPERIMENT == 'E103':
    model = module.BlendScorer(model, selection['alpha'])
print('Inference:', EXPERIMENT, chosen, 'RDKit', rdkit.__version__)

In [ ]:
REF = Path('/kaggle/working/cyril_reference')
build(str(TEST_PATH.parent / 'train.parquet'), str(REF))
sub = make_submission(model, str(TEST_PATH), str(REF), '/kaggle/working/submission.csv')
expected = pd.read_parquet(TEST_PATH, columns=['molecule_id']).molecule_id.drop_duplicates()
assert list(sub.columns) == ['molecule_id', 'smiles']
assert sub.molecule_id.is_unique and set(sub.molecule_id) == set(expected)
assert sub.smiles.notna().all()
assert sub.smiles.map(lambda s: 1 <= len(s.split(';')) <= 25 and all(s.split(';'))).all()
from rdkit import Chem
assert all(Chem.MolFromSmiles(s) is not None for row in sub.smiles for s in row.split(';'))
print('Validated submission:', sub.shape)
sub.head()